In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator, FixedLocator, FuncFormatter
from matplotlib.collections import LineCollection
from matplotlib import colors as mcolors
from snsphd.viz import phd_style
from util.faded_lines import draw_faded_hline
from util.pcr_loader import PcrLoader

colors, swatches = phd_style(jupyterStyle=True)

SHOW_DARK = False
ARROW = False
X_LIMITS = (0.023, 0.19)
RIGHT_TOP = 20e5
RIGHT_AXIS_COLOR = "#3054b8"

CURVE_SPECS = [
    {"label": r"15$\mu$m", "path": "../data/Low_Tc_Dev3_80nm_Wire/pcrcurve_cr_20241209-145103.obj"},
    {"label": r"18$\mu$m", "path": "../data/Low_Tc_Dev3_80nm_Wire/pcrcurve_cr_20241209-135210.obj"},
    {"label": r"29$\mu$m", "path": "../data/Low_Tc_Dev3_80nm_Wire/pcrcurve_cr_20241209-141437.obj"},
]

# Exact colours used in original plot_pcr (skip=0, first N entries)
PLASMA_LEVELS = [0.1, 0.5, 0.8, 1.0, 1.0, 0.99]
MARKERS       = ['o', '^', 'D', 'h']
MARKERS_DARK  = ['x', 'h', 'D', 'h']


In [ ]:
curves = [PcrLoader.from_pickle(spec["path"], label=spec["label"]) for spec in CURVE_SPECS]

fig, ax = plt.subplots(1, figsize=(7, 4))
ax_d = ax.twinx()
ax.set_zorder(1)
ax.patch.set_visible(False)

if SHOW_DARK:
    ax_d.spines['right'].set_visible(True)
    ax_d.spines['right'].set_linewidth(1.9)
    ax_d.spines['right'].set_color(RIGHT_AXIS_COLOR)
    ax_d.tick_params(axis='y', colors=RIGHT_AXIS_COLOR)
else:
    # Hide the spare twin axis when DCR is off.
    ax_d.spines['right'].set_visible(False)
    ax_d.get_yaxis().set_visible(False)

ax.grid(False)
ax_d.grid(False)

for i, (curve, spec) in enumerate(zip(curves, CURVE_SPECS)):
    color = plt.cm.plasma(PLASMA_LEVELS[i])
    if SHOW_DARK and curve.dark_counts is not None:
        ax_d.plot(
            curve.bias_uA, curve.dark_counts,
            MARKERS_DARK[i], c=RIGHT_AXIS_COLOR,
            lw=3, markersize=4, label=f"DCR {spec['label']}", alpha=0.6,
        )
    # Normalize each trace independently before overlaying.
    m = np.nanmax(curve.counts)
    ax.plot(
        curve.bias_uA, curve.counts / m,
        MARKERS[i], c=color,
        lw=3, markersize=4, label=spec['label'], alpha=0.6, linestyle='-',
    )

ax.set_xlabel(r"Bias Current ($\mu$A)", labelpad=10)
ax.set_xlim(X_LIMITS)
ax.set_ylabel("Normalized Counts (a.u.)", labelpad=15)

if SHOW_DARK:
    left_ymax = np.log10(RIGHT_TOP) / 6.0
    ax.set_ylim(0.0, left_ymax)
    ax.yaxis.set_major_locator(MultipleLocator(1/6))
    ax_d.set_yscale("log")
    ax_d.set_ylim(1, RIGHT_TOP)
    ax_d.set_ylabel("Dark Count Rate (cps)")
    ax_d.spines["right"].set_color(RIGHT_AXIS_COLOR)
    ax_d.tick_params(axis="y", colors=RIGHT_AXIS_COLOR)
    ax_d.yaxis.label.set_color(RIGHT_AXIS_COLOR)

    x0, x1 = ax.get_xlim()
    for y_lin in np.linspace(0, 1, 7):
        if y_lin <= left_ymax:
            xs = np.array([[x0, y_lin], [x1, y_lin]])
            lc = LineCollection([xs], colors=[mcolors.to_rgba("0.0", 0.25)], linewidths=0.5, zorder=0)
            ax.add_collection(lc)
    ymin, ymax_r = ax_d.get_ylim()
    for k in range(int(np.floor(np.log10(ymin))), int(np.floor(np.log10(ymax_r))) + 1):
        for m in range(2, 10):
            y_minor = m * (10 ** k)
            if ymin <= y_minor <= ymax_r:
                draw_faded_hline(ax_d, y_minor, x0, x1, n_segments=50,
                    base_color=RIGHT_AXIS_COLOR, alpha_max=0.2, lw=0.5, fade_from="right", zorder=0.05)
else:
    ax.grid()
    ax.set_ylim(bottom=0)

lines_pcr, labels_pcr = ax.get_legend_handles_labels()
lines_dcr, labels_dcr = ax_d.get_legend_handles_labels()
if lines_pcr:
    leg = ax.legend(lines_pcr, labels_pcr, loc='upper left', frameon=False, fontsize=11)
    ax.add_artist(leg)
if lines_dcr:
    ax_d.legend(lines_dcr, labels_dcr, loc='center right', frameon=False, fontsize=11)

if ARROW:
    ax.annotate("", xytext=(.43, 0.63), xy=(.49, 0.63),
        arrowprops=dict(arrowstyle="->", lw=3, mutation_scale=30, color="#c2cdea", alpha=1))
    leg_d = ax_d.get_legend()
    if leg_d is not None:
        leg_d.set_loc('lower right')

plt.tight_layout()

out_dir = Path("../out/wire")
out_dir.mkdir(parents=True, exist_ok=True)
st = "_arrow" if ARROW else ""
sr = "_with_darkcount" if SHOW_DARK else "_without_darkcount"
plt.savefig(out_dir / f"Low_Tc_Dev3_80nm_Wire{st}{sr}.png", dpi=300)
plt.savefig(out_dir / f"Low_Tc_Dev3_80nm_Wire{st}{sr}.pdf", dpi=300)
plt.show()
